In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

Mounted at /content/drive


In [2]:
DATA_PATH = "/content/drive/MyDrive/datasets/raw/"

In [3]:
df_fe = pd.read_csv(DATA_PATH + "application_train.csv")
print("Original shape:", df_fe.shape)
display(df_fe.head())

Original shape: (307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
# DAYS_EMPLOYED anomaly
df_fe["DAYS_EMPLOYED_ANOM"] = (df_fe["DAYS_EMPLOYED"] == 365243).astype(int)
df_fe["DAYS_EMPLOYED"] = df_fe["DAYS_EMPLOYED"].replace(365243, np.nan)

# EXT_SOURCE missingness
df_fe["EXT_SOURCE_1_MISSING"] = df_fe["EXT_SOURCE_1"].isna().astype(int)
df_fe["EXT_SOURCE_3_MISSING"] = df_fe["EXT_SOURCE_3"].isna().astype(int)

In [5]:
print("DAYS_EMPLOYED anomaly count:", df_fe["DAYS_EMPLOYED_ANOM"].sum())
print("Remaining 365243 values:", (df_fe["DAYS_EMPLOYED"] == 365243).sum())
print("EXT_SOURCE_1 missing count:", df_fe["EXT_SOURCE_1_MISSING"].sum())
print("EXT_SOURCE_3 missing count:", df_fe["EXT_SOURCE_3_MISSING"].sum())

DAYS_EMPLOYED anomaly count: 55374
Remaining 365243 values: 0
EXT_SOURCE_1 missing count: 173378
EXT_SOURCE_3 missing count: 60965


In [6]:
print("Zero AMT_INCOME_TOTAL:", (df_fe["AMT_INCOME_TOTAL"] == 0).sum())
print("Zero AMT_CREDIT:", (df_fe["AMT_CREDIT"] == 0).sum())
print("Zero AMT_ANNUITY:", (df_fe["AMT_ANNUITY"] == 0).sum())

Zero AMT_INCOME_TOTAL: 0
Zero AMT_CREDIT: 0
Zero AMT_ANNUITY: 0


In [7]:
#Age feature
df_fe["AGE_YEARS"] = -df_fe["DAYS_BIRTH"] / 365.25

In [9]:
#Employment years
df_fe["EMPLOYMENT_YEARS"] = -df_fe["DAYS_EMPLOYED"] / 365.25

In [11]:
#Financial ratios
df_fe["CREDIT_INCOME_RATIO"] = df_fe["AMT_CREDIT"] / df_fe["AMT_INCOME_TOTAL"]
df_fe["ANNUITY_INCOME_RATIO"] = df_fe["AMT_ANNUITY"] / df_fe["AMT_INCOME_TOTAL"]
df_fe["ANNUITY_CREDIT_RATIO"] = df_fe["AMT_ANNUITY"] / df_fe["AMT_CREDIT"]
df_fe["GOODS_CREDIT_RATIO"] = df_fe["AMT_GOODS_PRICE"] / df_fe["AMT_CREDIT"]

In [12]:
#Employment/age ratio
df_fe["EMPLOYMENT_AGE_RATIO"] = df_fe["EMPLOYMENT_YEARS"] / df_fe["AGE_YEARS"]

In [13]:
#Verify employment ratio NaNs
print("NaN in EMPLOYMENT_AGE_RATIO:", df_fe["EMPLOYMENT_AGE_RATIO"].isna().sum())
print("DAYS_EMPLOYED anomaly count:", df_fe["DAYS_EMPLOYED_ANOM"].sum())

NaN in EMPLOYMENT_AGE_RATIO: 55374
DAYS_EMPLOYED anomaly count: 55374


In [14]:
#EXT_SOURCE summary features

ext_cols = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
df_fe["EXT_SOURCE_MEAN"] = df_fe[ext_cols].mean(axis=1)
df_fe["EXT_SOURCE_MIN"] = df_fe[ext_cols].min(axis=1)
df_fe["EXT_SOURCE_MAX"] = df_fe[ext_cols].max(axis=1)
df_fe["EXT_SOURCE_STD"] = df_fe[ext_cols].std(axis=1)

In [15]:
#EXT_SOURCE interactions
df_fe["EXT_SOURCE_1_2"] = df_fe["EXT_SOURCE_1"] * df_fe["EXT_SOURCE_2"]
df_fe["EXT_SOURCE_1_3"] = df_fe["EXT_SOURCE_1"] * df_fe["EXT_SOURCE_3"]
df_fe["EXT_SOURCE_2_3"] = df_fe["EXT_SOURCE_2"] * df_fe["EXT_SOURCE_3"]

In [16]:
new_features = [
    "AGE_YEARS", "EMPLOYMENT_YEARS",
    "CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO",
    "ANNUITY_CREDIT_RATIO", "GOODS_CREDIT_RATIO",
    "EMPLOYMENT_AGE_RATIO",
    "EXT_SOURCE_MEAN", "EXT_SOURCE_MIN", "EXT_SOURCE_MAX", "EXT_SOURCE_STD",
    "EXT_SOURCE_1_2", "EXT_SOURCE_1_3", "EXT_SOURCE_2_3"
]

print("Number of new features:", len(new_features))
for f in new_features:
    print("-", f)

Number of new features: 14
- AGE_YEARS
- EMPLOYMENT_YEARS
- CREDIT_INCOME_RATIO
- ANNUITY_INCOME_RATIO
- ANNUITY_CREDIT_RATIO
- GOODS_CREDIT_RATIO
- EMPLOYMENT_AGE_RATIO
- EXT_SOURCE_MEAN
- EXT_SOURCE_MIN
- EXT_SOURCE_MAX
- EXT_SOURCE_STD
- EXT_SOURCE_1_2
- EXT_SOURCE_1_3
- EXT_SOURCE_2_3


In [17]:
display(df_fe[new_features].describe().T)

,count,mean,std,min,25%,50%,75%,max
AGE_YEARS,307511.0,43.906900,11.947950,2.050376e+01,33.984942,43.121150,53.886379,69.073238
EMPLOYMENT_YEARS,252137.0,6.527500,6.402081,-0.000000e+00,2.099932,4.511978,8.692676,49.040383
CREDIT_INCOME_RATIO,307511.0,3.957570,2.689728,4.807615e-03,2.018667,3.265067,5.159880,84.736842
ANNUITY_INCOME_RATIO,307499.0,0.180930,0.094574,2.238846e-04,0.114782,0.162833,0.229067,1.875965
ANNUITY_CREDIT_RATIO,307499.0,0.053695,0.022481,2.207258e-02,0.036900,0.050000,0.064043,0.124430
GOODS_CREDIT_RATIO,307233.0,0.900689,0.096630,1.666667e-01,0.834725,0.893815,1.000000,6.666667
EMPLOYMENT_AGE_RATIO,252137.0,0.156861,0.133549,-0.000000e+00,0.056099,0.118733,0.219170,0.728811
EXT_SOURCE_MEAN,307339.0,0.509251,0.149802,5.939651e-06,0.413648,0.524502,0.622819,0.878903
EXT_SOURCE_MIN,307339.0,0.399582,0.187425,8.173617e-08,0.253963,0.403167,0.553013,0.878903
EXT_SOURCE_MAX,307339.0,0.615882,0.156130,5.939651e-06,0.540654,0.648470,0.725276,0.962693


In [18]:
#Check inf in new features
new_numeric = df_fe[new_features].select_dtypes(include=np.number)
new_inf_count = np.isinf(new_numeric).sum().sum()

print("Inf in new features:", new_inf_count)

Inf in new features: 0


In [22]:
feature_target_corr = (
    df_fe[new_features + ["TARGET"]]
    .corr()["TARGET"]
    .drop("TARGET")
    .sort_values()
)

print("Correlation of engineered features with TARGET:")
display(feature_target_corr)

Correlation of engineered features with TARGET:


,TARGET
EXT_SOURCE_MEAN,-0.222052
EXT_SOURCE_2_3,-0.199487
EXT_SOURCE_MAX,-0.196876
EXT_SOURCE_1_3,-0.187832
EXT_SOURCE_MIN,-0.185266
EXT_SOURCE_1_2,-0.175575
AGE_YEARS,-0.078239
EMPLOYMENT_YEARS,-0.074958
EMPLOYMENT_AGE_RATIO,-0.067955
GOODS_CREDIT_RATIO,-0.065407


In [23]:
X_fe = df_fe.drop(columns=["TARGET", "SK_ID_CURR"])
y_fe = df_fe["TARGET"]

print("X shape:", X_fe.shape)
print("y shape:", y_fe.shape)

X shape: (307511, 137)
y shape: (307511,)


In [24]:
X_train_fe, X_valid_fe, y_train_fe, y_valid_fe = train_test_split(
    X_fe, y_fe, test_size=0.20, random_state=42, stratify=y_fe
)

print("Training shape:", X_train_fe.shape)
print("Validation shape:", X_valid_fe.shape)

Training shape: (246008, 137)
Validation shape: (61503, 137)


In [25]:
numeric_features_fe = X_train_fe.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features_fe = X_train_fe.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", len(numeric_features_fe))
print("Categorical features:", len(categorical_features_fe))

Numeric features: 121
Categorical features: 16


In [26]:
numeric_transformer_fe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer_fe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_fe = ColumnTransformer(transformers=[
    ("num", numeric_transformer_fe, numeric_features_fe),
    ("cat", categorical_transformer_fe, categorical_features_fe)
])

In [27]:
#XGBoost model (same parameters as baseline)
xgb_fe = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

In [28]:
xgb_fe_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_fe),
    ("model", xgb_fe)
])

In [29]:
print("Training XGBoost with application features...")
xgb_fe_pipeline.fit(X_train_fe, y_train_fe)
print("Training complete.")

Training XGBoost with application features...
Training complete.


In [30]:
xgb_fe_valid_proba = xgb_fe_pipeline.predict_proba(X_valid_fe)[:, 1]

In [31]:
xgb_fe_roc_auc = roc_auc_score(y_valid_fe, xgb_fe_valid_proba)
xgb_fe_pr_auc = average_precision_score(y_valid_fe, xgb_fe_valid_proba)

print("XGBOOST + APPLICATION FEATURES")
print(f"ROC-AUC: {xgb_fe_roc_auc:.4f}")
print(f"PR-AUC:  {xgb_fe_pr_auc:.4f}")

XGBOOST + APPLICATION FEATURES
ROC-AUC: 0.7694
PR-AUC:  0.2627


In [32]:
baseline_roc_auc = 0.7612
baseline_pr_auc = 0.2516

roc_change = xgb_fe_roc_auc - baseline_roc_auc
pr_change = xgb_fe_pr_auc - baseline_pr_auc

print("IMPROVEMENT OVER BASELINE")
print(f"ROC-AUC change: {roc_change:+.4f}")
print(f"PR-AUC change:  {pr_change:+.4f}")

IMPROVEMENT OVER BASELINE
ROC-AUC change: +0.0082
PR-AUC change:  +0.0111


In [33]:
application_feature_result = pd.DataFrame({
    "Experiment": ["XGBoost + Application Features"],
    "ROC-AUC": [xgb_fe_roc_auc],
    "PR-AUC": [xgb_fe_pr_auc],
    "ROC-AUC Change": [roc_change],
    "PR-AUC Change": [pr_change]
})

display(application_feature_result.style.format({
    "ROC-AUC": "{:.4f}",
    "PR-AUC": "{:.4f}",
    "ROC-AUC Change": "{:+.4f}",
    "PR-AUC Change": "{:+.4f}"
}))

,Experiment,ROC-AUC,PR-AUC,ROC-AUC Change,PR-AUC Change
0,XGBoost + Application Features,0.7694,0.2627,+0.0082,+0.0111


In [35]:
!pip install mlflow -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 110.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.2/216.2 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [37]:
import mlflow

mlflow.set_tracking_uri("sqlite:////content/drive/MyDrive/RupeeRisk/mlflow.db")
mlflow.set_experiment("RupeeRisk")

with mlflow.start_run(run_name="Logistic_Regression_Baseline"):
    mlflow.log_param("stage", "MODEL01 - Baseline")
    mlflow.log_metric("roc_auc", 0.7501)
    mlflow.log_metric("pr_auc", 0.2326)

with mlflow.start_run(run_name="XGBoost_Baseline"):
    mlflow.log_param("stage", "MODEL01 - Baseline")
    mlflow.log_metric("roc_auc", 0.7612)
    mlflow.log_metric("pr_auc", 0.2516)

with mlflow.start_run(run_name="XGBoost_scale_pos_weight"):
    mlflow.log_param("stage", "MODEL01 - Baseline")
    mlflow.log_param("outcome", "Tested and rejected - slightly hurt both metrics")
    mlflow.log_metric("roc_auc", 0.7600)
    mlflow.log_metric("pr_auc", 0.2493)

print("MODEL01 baseline runs backfilled into MLflow.")

2026/08/22 10:08:11 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/22 10:08:11 INFO mlflow.store.db.utils: Updating database tables
2026/08/22 10:08:16 INFO mlflow.tracking.fluent: Experiment with name 'RupeeRisk' does not exist. Creating a new experiment.


MODEL01 baseline runs backfilled into MLflow.


In [41]:
with mlflow.start_run(run_name="XGBoost_Application_Features"):

    # Experiment information
    mlflow.log_param("stage", "MODEL02 - Application Feature Engineering")
    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("n_new_features", len(new_features))
    mlflow.log_param("features_added", ", ".join(new_features))

    # XGBoost hyperparameters (so this run is fully reproducible later)
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("subsample", 0.8)
    mlflow.log_param("colsample_bytree", 0.8)
    mlflow.log_param("scale_pos_weight", False)

    # Metrics
    mlflow.log_metric("roc_auc", xgb_fe_roc_auc)
    mlflow.log_metric("pr_auc", xgb_fe_pr_auc)
    mlflow.log_metric("roc_auc_change_vs_baseline", roc_change)
    mlflow.log_metric("pr_auc_change_vs_baseline", pr_change)

print("MODEL02 experiment logged to MLflow.")

MODEL02 experiment logged to MLflow.


In [42]:
runs = mlflow.search_runs(experiment_names=["RupeeRisk"])
display(runs[["tags.mlflow.runName", "metrics.roc_auc", "metrics.pr_auc"]])

,tags.mlflow.runName,metrics.roc_auc,metrics.pr_auc
0,XGBoost_Application_Features,0.769403,0.262725
1,XGBoost_Application_Features,0.769403,0.262725
2,XGBoost_scale_pos_weight,0.760000,0.249300
3,XGBoost_Baseline,0.761200,0.251600
4,Logistic_Regression_Baseline,0.750100,0.232600


In [43]:
RESULTS_PATH = "/content/drive/MyDrive/RupeeRisk/"
os.makedirs(RESULTS_PATH, exist_ok=True)

application_feature_result.to_csv(RESULTS_PATH + "application_feature_experiment.csv", index=False)
print("Experiment saved to:", RESULTS_PATH + "application_feature_experiment.csv")

Experiment saved to: /content/drive/MyDrive/RupeeRisk/application_feature_experiment.csv


In [44]:
runs = mlflow.search_runs(experiment_names=["RupeeRisk"])
display(runs[["run_id", "tags.mlflow.runName", "metrics.roc_auc", "metrics.pr_auc", "start_time"]])

,run_id,tags.mlflow.runName,metrics.roc_auc,metrics.pr_auc,start_time
0,fab91375274f4180a02d7d5407f0239b,XGBoost_Application_Features,0.769403,0.262725,2026-08-22 10:37:14.948000+00:00
1,68991e7954e148639e6b110ce7ee7118,XGBoost_Application_Features,0.769403,0.262725,2026-08-22 10:09:32.695000+00:00
2,4efc6ca8dcb0436ca7a68466ea59a8b3,XGBoost_scale_pos_weight,0.760000,0.249300,2026-08-22 10:08:16.958000+00:00
3,fba077fde8244f718c60ddb06376a24f,XGBoost_Baseline,0.761200,0.251600,2026-08-22 10:08:16.831000+00:00
4,c8217dca421d439f9f3097348f2331a0,Logistic_Regression_Baseline,0.750100,0.232600,2026-08-22 10:08:16.663000+00:00


In [45]:
mlflow.delete_run("68991e7954e148639e6b110ce7ee7118")

In [46]:
runs = mlflow.search_runs(experiment_names=["RupeeRisk"])
display(runs[["tags.mlflow.runName", "metrics.roc_auc", "metrics.pr_auc"]])

,tags.mlflow.runName,metrics.roc_auc,metrics.pr_auc
0,XGBoost_Application_Features,0.769403,0.262725
1,XGBoost_scale_pos_weight,0.760000,0.249300
2,XGBoost_Baseline,0.761200,0.251600
3,Logistic_Regression_Baseline,0.750100,0.232600
